# GRIT-Echo — GRanular Information Tracking in Echocardiography

In [1]:
# ============================================================
# STEP 1 - Environment: installs, imports, seeds, paths
# ============================================================
import importlib, subprocess, sys

def ensure(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg],
                       check=True)
    return pkg

for pkg, name in [("albumentations", None), ("nibabel", None), ("gymnasium", None),
                  ("segmentation_models_pytorch", "segmentation-models-pytorch"),
                  ("stable_baselines3", "stable-baselines3")]:
    ensure(pkg, name)

import json, os, platform, random, warnings
from itertools import combinations
from pathlib import Path
warnings.filterwarnings("ignore")

import albumentations as A
import cv2
import gymnasium as gym
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from albumentations.pytorch import ToTensorV2
from gymnasium import spaces
from matplotlib.colors import ListedColormap
from scipy import ndimage, stats
from scipy.spatial.distance import cdist
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def find_dir(candidates):
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"None of these exist: {candidates}")

BASE = find_dir([Path("/kaggle/input/datasets/brainsect/camus-v1"),
                 Path("/kaggle/input/camus-v1")])
NIFTI = BASE / "database_nifti"
SPLIT = BASE / "database_split"
SEQ = find_dir([Path("/kaggle/input/datasets/brainsect/camus-half-sequences"),
                Path("/kaggle/input/camus-half-sequences")])
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
WORK.mkdir(parents=True, exist_ok=True)
CKPT = WORK / "unet_resnet34_camus.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = device.type == "cuda"          # mixed precision: ~2x faster on the T4

TARGET_SIZE = (256, 256)
INTENSITY_PCTS = (1, 99)
MAX_ENTROPY_BITS = float(np.log2(4))
VIEWS, INSTANTS = ("2CH", "4CH"), ("ED", "ES")
FG_CLASSES = (1, 2, 3)

def fingerprint():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return (f"python {platform.python_version()} | torch {torch.__version__} | "
            f"numpy {np.__version__} | {gpu} | AMP={AMP}")

print(fingerprint())
print(f"NIFTI={NIFTI}\nSPLIT={SPLIT}\nSEQ={SEQ}\nWORK={WORK}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


python 3.12.13 | torch 2.10.0+cu128 | numpy 2.0.2 | Tesla T4 | AMP=True
NIFTI=/kaggle/input/datasets/brainsect/camus-v1/database_nifti
SPLIT=/kaggle/input/datasets/brainsect/camus-v1/database_split
SEQ=/kaggle/input/datasets/brainsect/camus-half-sequences
WORK=/kaggle/working


In [2]:
# ============================================================
# STEP 2 - Cohort audit, official splits, metadata
# ============================================================
def frame_path(patient_dir, view, instant, mask=False):
    """Tolerant NIfTI locator: accepts .nii and .nii.gz; raises if absent."""
    suffix = "_gt" if mask else ""
    stem = f"{patient_dir.name}_{view}_{instant}{suffix}"
    matches = sorted(patient_dir.glob(f"{stem}.nii*"))
    if not matches:
        raise FileNotFoundError(f"Missing {'mask' if mask else 'image'}: {patient_dir.name}/{stem}")
    return matches[0]

def cohort_audit_table(nifti_dir):
    """One row per frame; headers only (nibabel defers voxel decompression)."""
    rows = []
    for patient in sorted(p for p in nifti_dir.iterdir() if p.is_dir()):
        for view in VIEWS:
            for instant in INSTANTS:
                img = nib.load(frame_path(patient, view, instant))
                nib.load(frame_path(patient, view, instant, mask=True))
                sx, sy = img.header.get_zooms()[:2]
                rows.append({"patient": patient.name, "view": view, "instant": instant,
                             "h": img.shape[0], "w": img.shape[1],
                             "sx_mm": float(sx), "sy_mm": float(sy)})
    return pd.DataFrame(rows)

def read_subgroup(filename):
    path = SPLIT / filename
    assert path.exists(), f"missing {path}"
    return [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]

train_ids = read_subgroup("subgroup_training.txt")
val_ids = read_subgroup("subgroup_validation.txt")
test_ids = read_subgroup("subgroup_testing.txt")
train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)
# fail fast on duplicates / leakage
assert len(train_ids) == len(train_set) and len(val_ids) == len(val_set) \
    and len(test_ids) == len(test_set), "Duplicate identifiers inside a subgroup file."
assert not (train_set & test_set) and not (val_set & test_set), "Test leakage."

REQUIRED_CFG_KEYS = ("ED", "ES", "NbFrame", "Sex", "Age", "ImageQuality", "EF", "FrameRate")

def parse_cfg(path):
    cfg = {}
    for line in path.read_text().splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            cfg[k.strip()] = v.strip()
    missing = [k for k in REQUIRED_CFG_KEYS if k not in cfg]
    assert not missing, f"{path.name} missing {missing}"
    return cfg

rows = []
for pid in sorted(train_set | val_set | test_set):
    for view in VIEWS:
        c = parse_cfg(NIFTI / pid / f"Info_{view}.cfg")
        rows.append({"patient": pid, "view": view, "ed_frame": int(c["ED"]),
                     "es_frame": int(c["ES"]), "quality": c["ImageQuality"],
                     "ref_ef": float(c["EF"]), "sex": c["Sex"], "age": int(c["Age"])})
meta = pd.DataFrame(rows)

audit_table = cohort_audit_table(NIFTI)
print(f"{audit_table.patient.nunique()} patients | {len(audit_table)} frames | "
      f"train/val/test = {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")
print("quality per view:")
print(meta.groupby(["view", "quality"]).size().unstack(fill_value=0))

500 patients | 2000 frames | train/val/test = 400/50/50
quality per view:
quality  Good  Medium  Poor
view                       
2CH       217     214    69
4CH       288     165    47


In [ ]:
# ============================================================
# STEP 3 - Transforms, dataset, loaders  (letterbox fix)
# ============================================================
def normalize_intensity(raw, percentiles=INTENSITY_PCTS):
    """Percentile-clip then rescale to [0, 255]; identical at train and inference."""
    lo, hi = np.percentile(raw, percentiles)
    span = max(hi - lo, 1e-6)
    return ((np.clip(raw, lo, hi) - lo) / span * 255.0).astype(np.uint8)

def pad_to_square(img, **kwargs):
    """Letterbox-pad to square so Resize never distorts cardiac geometry.

    **kwargs: albumentations 2.x passes extras (e.g. 'shape') to A.Lambda
    callables; accepting them keeps both 2-D masks and HxWx3 RGB working."""
    h, w = img.shape[:2]
    if h == w:
        return img
    pad = abs(h - w)
    tail = ((0, 0),) * (img.ndim - 2)      # pad channel axis too, if present
    if h < w:
        t, b = pad // 2, pad - pad // 2
        return np.pad(img, ((t, b), (0, 0)) + tail)
    l, r = pad // 2, pad - pad // 2
    return np.pad(img, ((0, 0), (l, r)) + tail)

def base_tf():
    return [A.Lambda(image=pad_to_square, mask=pad_to_square, p=1.0),
            A.Resize(height=TARGET_SIZE[0], width=TARGET_SIZE[1],
                     interpolation=cv2.INTER_AREA, mask_interpolation=cv2.INTER_NEAREST),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()]

def build_transform(split):
    train_only = [A.HorizontalFlip(p=0.5),
                  A.Affine(rotate=(-12, 12), scale=(0.92, 1.08),
                           translate_percent=(-0.05, 0.05), interpolation=cv2.INTER_LINEAR,
                           mask_interpolation=cv2.INTER_NEAREST,
                           border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0, p=0.6),
                  A.RandomBrightnessContrast(p=0.4)]
    return A.Compose((train_only if split == "train" else []) + base_tf())

seq_tf = A.Compose(base_tf())

class CamusDataset(Dataset):
    def __init__(self, ids, split):
        self.samples = [(pid, v, i) for pid in ids for v in VIEWS for i in INSTANTS]
        self.transform = build_transform(split)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, view, instant = self.samples[idx]
        img = np.squeeze(np.asarray(
            nib.load(frame_path(NIFTI / pid, view, instant)).dataobj)).astype(np.float32)
        msk = np.squeeze(np.asarray(
            nib.load(frame_path(NIFTI / pid, view, instant, mask=True)).dataobj)).astype(np.uint8)
        rgb = np.stack([normalize_intensity(img)] * 3, axis=-1)
        out = self.transform(image=rgb, mask=msk)
        return out["image"], out["mask"].long()

def worker_init(worker_id):
    seed = torch.initial_seed() % 2**32
    random.seed(seed); np.random.seed(seed)

def make_loader(ids, split):
    gen = torch.Generator().manual_seed(SEED)
    return DataLoader(CamusDataset(ids, split), batch_size=16,
                      shuffle=(split == "train"), num_workers=2,
                      pin_memory=AMP, worker_init_fn=worker_init,
                      generator=gen, persistent_workers=True)

train_loader = make_loader(train_ids, "train")
val_loader = make_loader(val_ids, "val")
print(f"train: {len(train_loader)} batches | val: {len(val_loader)} batches")

train: 100 batches | val: 13 batches


In [ ]:
# ============================================================
# STEP 4 - U-Net training  (AMP + grad clip + early stop, auto-safe)
# ============================================================
RETRAIN = "auto"   # True: force retrain | False: force load | "auto": train if no checkpoint
EPOCHS, LR, WEIGHT_DECAY = 20, 1e-3, 1e-4
PATIENCE = 6

dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
ce_loss = CrossEntropyLoss()

def criterion(logits, targets):
    if targets.ndim == 4:
        targets = targets.squeeze(1)
    return dice_loss(logits, targets) + ce_loss(logits, targets.long())

def foreground_dice(logits, targets):
    """Macro Dice over the three anatomical classes (model-selection metric)."""
    pred = logits.argmax(1)
    per = []
    for c in FG_CLASSES:
        inter = ((pred == c) & (targets == c)).sum().float()
        denom = (pred == c).sum().float() + (targets == c).sum().float()
        per.append((2.0 * inter / denom.clamp(min=1.0)).item())
    return float(np.mean(per))

train_flag = (RETRAIN is True) or (RETRAIN == "auto" and not CKPT.exists())

if train_flag:
    model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                     in_channels=3, classes=4).to(device)
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler(enabled=AMP)      
    history, best_dice, best_epoch, bad = [], -1.0, -1, 0
    for epoch in range(EPOCHS):
        model.train(); t_tot = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=AMP):
                loss = criterion(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
            scaler.step(opt); scaler.update()
            t_tot += loss.item() * len(xb)
        model.eval(); v_tot, dices = 0.0, []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                with torch.amp.autocast("cuda", enabled=AMP):
                    lg = model(xb)
                v_tot += criterion(lg.float(), yb).item() * len(xb)
                dices.append(foreground_dice(lg, yb))
        sched.step()
        rec = {"epoch": epoch + 1, "train_loss": t_tot / len(train_loader.dataset),
               "val_loss": v_tot / len(val_loader.dataset),
               "val_dice": float(np.mean(dices))}
        history.append(rec)
        mark = ""
        if rec["val_dice"] > best_dice:
            best_dice, best_epoch, bad = rec["val_dice"], epoch + 1, 0
            torch.save(model.state_dict(), CKPT); mark = "  <- saved"
        else:
            bad += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | train {rec['train_loss']:.4f} | "
              f"val {rec['val_loss']:.4f} | valDice {rec['val_dice']:.4f}{mark}")
        if bad >= PATIENCE:                       # fix #6: early stopping
            print(f"early stop at epoch {epoch+1}"); break
    (WORK / "history.json").write_text(json.dumps(history, indent=2))
    model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
    print(f"best val Dice {best_dice:.4f} @ epoch {best_epoch}")
else:
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                     in_channels=3, classes=4).to(device)
    model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
    print(f"loaded canonical checkpoint {CKPT.name}")
model.eval()

loaded canonical checkpoint unet_resnet34_camus.pt


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [5]:
# ============================================================
# STEP 5 - Test evaluation: Dice + HD95 (predictions computed ONCE, cached)
# ============================================================
def predict_patient(pid):
    """Native-resolution argmax masks for all four frames of one patient.

    Geometry is handled exactly: pad raw to square -> resize 256 -> predict ->
    resize back to the padded square -> crop the letterbox padding.
    """
    tf = build_transform("test")
    preds = {}
    with torch.no_grad():
        for view in VIEWS:
            for instant in INSTANTS:
                raw = np.squeeze(np.asarray(
                    nib.load(frame_path(NIFTI / pid, view, instant)).dataobj)).astype(np.float32)
                h, w = raw.shape
                rgb = np.stack([normalize_intensity(pad_to_square(raw))] * 3, axis=-1)
                tens = tf(image=rgb)["image"].unsqueeze(0).to(device)
                pred = model(tens).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
                side = max(h, w)
                big = cv2.resize(pred, (side, side), interpolation=cv2.INTER_NEAREST)
                if h < w:
                    top = (w - h) // 2
                    big = big[top:top + h, :]
                elif w < h:
                    left = (h - w) // 2
                    big = big[:, left:left + w]
                preds[(view, instant)] = big
    return preds

def get_test_predictions(ids):
    """Fix #3: run the test split once and cache; reuse for LVEF below."""
    cache = WORK / "pred_cache.npz"
    if cache.exists():
        blob = np.load(cache)
        out = {}
        for k in blob.files:
            pid, v, i = k.split("__")
            out.setdefault(pid, {})[(v, i)] = blob[k]
        return out
    allp = {pid: predict_patient(pid) for pid in ids}
    np.savez_compressed(cache, **{f"{pid}__{v}__{i}": m
                                  for pid, d in allp.items() for (v, i), m in d.items()})
    return allp

def dice_score(pred, gt, c):
    inter = ((pred == c) & (gt == c)).sum()
    denom = (pred == c).sum() + (gt == c).sum()
    return float(2 * inter / denom) if denom > 0 else np.nan

def hd95_mm(pred, gt, c, spacing):
    """95th-percentile symmetric boundary distance in millimetres."""
    def boundary(m):
        m = m.astype(bool)
        return np.argwhere(m & ~ndimage.binary_erosion(m)) if m.any() else np.empty((0, 2), int)
    pb, gb = boundary(pred == c), boundary(gt == c)
    if len(pb) == 0 or len(gb) == 0:
        return np.nan
    scale = np.asarray(spacing, float)
    d = cdist(pb * scale, gb * scale)
    return float(max(np.percentile(d.min(axis=1), 95), np.percentile(d.min(axis=0), 95)))

test_preds = get_test_predictions(test_ids)
records = []
for pid in test_ids:
    for (view, instant), pm in test_preds[pid].items():
        img = nib.load(frame_path(NIFTI / pid, view, instant, mask=True))
        gm = np.squeeze(np.asarray(img.dataobj)).astype(np.uint8)
        spacing = img.header.get_zooms()[:2]
        for c in FG_CLASSES:
            records.append({"patient": pid, "view": view, "instant": instant,
                            "class": c, "dice": dice_score(pm, gm, c),
                            "hd95_mm": hd95_mm(pm, gm, c, spacing)})
test_metrics = pd.DataFrame(records)
print(test_metrics.groupby("class")[["dice", "hd95_mm"]].agg(["mean", "std"]).round(3))

        dice        hd95_mm       
        mean    std    mean    std
class                             
1      0.935  0.031   3.924  1.913
2      0.877  0.039   4.171  1.937
3      0.914  0.058   4.307  3.254


In [6]:
# ============================================================
# STEP 6 - Clinical endpoint: Simpson biplane LVEF (ASE 2015)
# ============================================================
def largest_component(binary):
    labs, n = ndimage.label(binary)
    if n <= 1:
        return binary
    sizes = ndimage.sum(binary, labs, index=np.arange(1, n + 1))
    return labs == int(np.argmax(sizes)) + 1

def cavity_profile(mask, n_disks=20):
    blob = largest_component(mask == 1)
    ys, xs = np.nonzero(blob)
    if len(xs) < 20:
        return np.zeros(n_disks), 0.0
    pts = np.stack([xs, ys], 1).astype(float); pts -= pts.mean(0)
    _, _, vt = np.linalg.svd(pts, full_matrices=False)
    proj, orth = pts @ vt[0], pts @ vt[1]
    edges = np.linspace(proj.min(), proj.max(), n_disks + 1)
    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        sel = (proj >= a) & (proj < b)
        widths.append(float(orth[sel].max() - orth[sel].min()) if sel.any() else 0.0)
    return np.array(widths), float(proj.max() - proj.min())

def volume_ml(widths_px, length_px, spacing_mm, partner_px=None):
    mm = float(np.mean(spacing_mm))
    h_cm = length_px * mm / 10.0 / len(widths_px)
    d1 = widths_px * mm / 10.0
    if partner_px is None:
        return float((np.pi / 4) * np.sum(d1**2) * h_cm)
    d2 = partner_px * mm / 10.0
    return float((np.pi / 4) * np.sum(d1 * d2) * h_cm)

def load_mask_spacing(pid, view, instant):
    img = nib.load(frame_path(NIFTI / pid, view, instant, mask=True))
    return np.squeeze(np.asarray(img.dataobj)), img.header.get_zooms()[:2]

def ef(edv, esv):
    return (edv - esv) / edv * 100.0 if edv > 0 else np.nan

def biplane_from(mask_getter):
    """Biplane ED/ES volumes from mask_getter(view, instant) -> (mask, spacing)."""
    prof = {}
    for view in VIEWS:
        for instant in INSTANTS:
            m, sp = mask_getter(view, instant)
            w, l = cavity_profile(m)
            prof[(view, instant)] = (w, l, sp)
    bp = {}
    for instant in INSTANTS:
        (w1, l1, s1), (w2, l2, _) = prof[("2CH", instant)], prof[("4CH", instant)]
        bp[instant] = volume_ml(w1, (l1 + l2) / 2, s1, partner_px=w2)
    return bp

ref_ef = {(r.patient, r.view): r.ref_ef for r in meta.itertuples()}
recs = []
for pid in test_ids:
    bp_pred = biplane_from(
        lambda v, i, _pid=pid: (test_preds[_pid][(v, i)], load_mask_spacing(_pid, v, i)[1]))
    bp_exp = biplane_from(lambda v, i, _pid=pid: load_mask_spacing(_pid, v, i))
    recs.append({"patient": pid,
                 "ef_pred": ef(bp_pred["ED"], bp_pred["ES"]),
                 "ef_expert_masks": ef(bp_exp["ED"], bp_exp["ES"]),
                 "ref_ef": ref_ef[(pid, "2CH")]})
ef_test = pd.DataFrame(recs)
d_mask, d_ref = ef_test.ef_pred - ef_test.ef_expert_masks, ef_test.ef_pred - ef_test.ref_ef
print(f"pred vs expert-mask EF | MAE {d_mask.abs().mean():.2f} | bias {d_mask.mean():+.2f} | "
      f"r {ef_test.ef_pred.corr(ef_test.ef_expert_masks):.3f}")
print(f"pred vs clinical ref  | MAE {d_ref.abs().mean():.2f} | bias {d_ref.mean():+.2f} | "
      f"r {ef_test.ef_pred.corr(ef_test.ref_ef):.3f}")

pred vs expert-mask EF | MAE 4.38 | bias -2.33 | r 0.895
pred vs clinical ref  | MAE 7.17 | bias +5.49 | r 0.864


In [7]:
# ============================================================
# STEP 7 - Frame-wise entropy + granule calibration (ONE function, no copy-paste)
# ============================================================
def load_half_sequence(pid, view):
    matches = sorted(SEQ.glob(f"{pid}_{view}_half_sequence.nii*"))
    if not matches:
        raise FileNotFoundError(f"Missing half-sequence for {pid} {view}")
    return np.squeeze(nib.load(matches[0]).get_fdata(dtype=np.float32))

@torch.no_grad()
def extract_entropy(ids):
    """Fix #4: single implementation used for BOTH val (calibration) and test."""
    records = []
    for pid in ids:
        for view in VIEWS:
            try:
                vol = load_half_sequence(pid, view)
            except FileNotFoundError:
                continue
            std = [normalize_intensity(vol[..., t]) for t in range(vol.shape[-1])]
            tens = torch.stack([seq_tf(image=np.stack([s] * 3, -1))["image"] for s in std]).to(device)
            probs = torch.softmax(model(tens), 1)
            ent = (-(probs * torch.log2(probs + 1e-7)).sum(1) / MAX_ENTROPY_BITS).cpu().numpy()
            fg = probs.argmax(1).cpu().numpy() > 0
            for t in range(vol.shape[-1]):
                tissue = ent[t][fg[t]]
                records.append({"patient": pid, "view": view, "frame_idx": t,
                                "global_entropy": float(ent[t].mean()),
                                "tissue_entropy": float(tissue.mean()) if tissue.size else np.nan})
    df = pd.DataFrame(records)
    assert len(df) > 0, "No entropy rows: check SEQ dataset mounting."
    return df

GRANULE_MAP = {"crisp": 0, "vague": 1, "artifact-prone": 2}

val_entropy = extract_entropy(val_ids)
GRANULE_CUTS = tuple(np.quantile(val_entropy.tissue_entropy.dropna(), [1 / 3, 2 / 3]))

def granule_of(e):
    if e <= GRANULE_CUTS[0]:
        return "crisp"
    return "vague" if e <= GRANULE_CUTS[1] else "artifact-prone"

def assign_granules(df):
    out = df.copy()
    out["granule"] = out.tissue_entropy.map(granule_of)
    return out

print(f"frozen granule cutpoints (val terciles): {GRANULE_CUTS}")
test_entropy = extract_entropy(test_ids).merge(
    meta[["patient", "view", "quality"]], on=["patient", "view"], how="left")
print(f"val frames {len(val_entropy)} | test frames {len(test_entropy)}")

frozen granule cutpoints (val terciles): (np.float64(0.0788843110203743), np.float64(0.08814053237438202))
val frames 1858 | test frames 1968


In [8]:
# ============================================================
# STEP 8 - RQ1: entropy vs image quality (Kruskal-Wallis + post-hoc)
# ============================================================
QUALITY_ORDER = ["Good", "Medium", "Poor"]
GRANULE_ORDER = ["crisp", "vague", "artifact-prone"]

# (patient, view) is the independent unit; frame rows would pseudoreplicate
seq_ent = (test_entropy.groupby(["patient", "view", "quality"])
           .agg(tissue_entropy=("tissue_entropy", "mean")).reset_index())
groups = [seq_ent.loc[seq_ent.quality == q, "tissue_entropy"].dropna().values
          for q in QUALITY_ORDER]
H, p_kw = stats.kruskal(*groups)
N, K = len(seq_ent), len(QUALITY_ORDER)
eps_sq, eta_sq = H / (N - 1), (H - K + 1) / (N - K)
print(f"Kruskal-Wallis (N={N}, k={K}): H={H:.2f}, p={p_kw:.2e}, eps^2={eps_sq:.3f}")

seq_gran = assign_granules(seq_ent)
for a, b in combinations(QUALITY_ORDER, 2):
    xa = seq_gran.loc[seq_gran.quality == a, "tissue_entropy"].dropna()
    xb = seq_gran.loc[seq_gran.quality == b, "tissue_entropy"].dropna()
    U, p = stats.mannwhitneyu(xa, xb, alternative="two-sided")
    print(f"  {a} vs {b}: U={U:.1f}, p_adj={min(p * 3, 1.0):.2e}, "
          f"r_rb={abs(1 - 2 * U / (len(xa) * len(xb))):.3f}")

comp = pd.crosstab(seq_gran.quality, seq_gran.granule).reindex(
    index=QUALITY_ORDER, columns=GRANULE_ORDER).fillna(0)
print(comp.div(comp.sum(axis=1), axis=0).round(3))

Kruskal-Wallis (N=100, k=3): H=21.99, p=1.68e-05, eps^2=0.222
  Good vs Medium: U=522.0, p_adj=3.17e-03, r_rb=0.415
  Good vs Poor: U=102.0, p_adj=1.19e-04, r_rb=0.711
  Medium vs Poor: U=180.0, p_adj=1.17e-01, r_rb=0.368
granule  crisp  vague  artifact-prone
quality                              
Good     0.362  0.383           0.255
Medium   0.105  0.447           0.447
Poor     0.000  0.267           0.733


In [9]:
# ============================================================
# STEP 9 - RQ2: granule burden predicts error; gating; risk-coverage
# ============================================================
seq_stats = (assign_granules(test_entropy)
             .groupby(["patient", "view", "quality"])
             .agg(artifact_frac=("granule", lambda s: (s == "artifact-prone").mean()),
                  crisp_frac=("granule", lambda s: (s == "crisp").mean()),
                  mean_tissue_entropy=("tissue_entropy", "mean"))
             .reset_index())
seg_err = (test_metrics.groupby(["patient", "view"])
           .agg(mean_dice=("dice", "mean"), mean_hd95=("hd95_mm", "mean"))
           .reset_index())
rq2 = seq_stats.merge(seg_err, on=["patient", "view"]).dropna(subset=["mean_dice", "mean_hd95"])

rho_dice, p_dice = stats.spearmanr(rq2.artifact_frac, rq2.mean_dice)
rho_hd95, p_hd95 = stats.spearmanr(rq2.artifact_frac, rq2.mean_hd95)
print(f"artifact_frac vs mean Dice : rho={rho_dice:+.3f} (p={p_dice:.2e})")
print(f"artifact_frac vs mean HD95 : rho={rho_hd95:+.3f} (p={p_hd95:.2e})")

pat = (rq2.groupby("patient")
       .agg(artifact_frac=("artifact_frac", "max"), crisp_frac=("crisp_frac", "min"))
       .reset_index().merge(ef_test, on="patient"))
pat["abs_ef_err"] = (pat.ef_pred - pat.ef_expert_masks).abs()
rho_ef, p_ef = stats.spearmanr(pat.artifact_frac, pat.abs_ef_err)
print(f"worst-view artifact_frac vs |EF error| : rho={rho_ef:+.3f} (p={p_ef:.2e})")

rq2_s = rq2.sort_values("artifact_frac", ascending=True, kind="stable")
for cov in (1.00, 0.90, 0.80, 0.70):
    kept = rq2_s.head(int(round(cov * len(rq2_s))))
    print(f"  coverage {cov:.2f} | n={len(kept):3d} | "
          f"Dice {kept.mean_dice.mean():.3f} | HD95 {kept.mean_hd95.mean():.2f} mm")

rq2["flagged"] = rq2.artifact_frac > 0.5
gate = rq2.groupby("flagged").agg(n=("mean_dice", "size"), mean_dice=("mean_dice", "mean"),
                                  mean_hd95=("mean_hd95", "mean"))
print(gate.round(3))
u_stat, p_flag = stats.mannwhitneyu(rq2.loc[~rq2.flagged, "mean_dice"],
                                    rq2.loc[rq2.flagged, "mean_dice"], alternative="greater")
print(f"retained > flagged: p={p_flag:.2e}")

artifact_frac vs mean Dice : rho=-0.543 (p=5.37e-09)
artifact_frac vs mean HD95 : rho=+0.393 (p=5.18e-05)
worst-view artifact_frac vs |EF error| : rho=+0.289 (p=4.20e-02)
  coverage 1.00 | n=100 | Dice 0.909 | HD95 4.13 mm
  coverage 0.90 | n= 90 | Dice 0.910 | HD95 4.08 mm
  coverage 0.80 | n= 80 | Dice 0.912 | HD95 4.00 mm
  coverage 0.70 | n= 70 | Dice 0.916 | HD95 3.85 mm
          n  mean_dice  mean_hd95
flagged                          
False    61      0.919      3.677
True     39      0.893      4.848
retained > flagged: p=1.03e-07


In [10]:
# ============================================================
# STEP 10 - Consolidated results summary
# ============================================================
tm = test_metrics.groupby("class")[["dice", "hd95_mm"]].mean()
summary = pd.DataFrame([
    ("Segmentation (test)", "Dice LV/MY/RV",
     " / ".join(f"{tm.loc[c, 'dice']:.3f}" for c in (1, 2, 3))),
    ("Segmentation (test)", "HD95 mm LV/MY/RV",
     " / ".join(f"{tm.loc[c, 'hd95_mm']:.2f}" for c in (1, 2, 3))),
    ("LVEF (test)", "MAE vs expert / vs ref (EF%)",
     f"{d_mask.abs().mean():.2f} / {d_ref.abs().mean():.2f}"),
    ("RQ1 (test)", "Kruskal-Wallis H, p, eps^2", f"{H:.2f}, {p_kw:.2e}, {eps_sq:.3f}"),
    ("RQ2 (test)", "Spearman rho(artifact granule, Dice), p",
     f"{rho_dice:+.3f}, {p_dice:.2e}"),
], columns=["block", "metric", "value"])
display(summary)
summary.to_csv(WORK / "results_summary.csv", index=False)
print("saved -> results_summary.csv")


,block,metric,value
0,Segmentation (test),Dice LV/MY/RV,0.935 / 0.877 / 0.914
1,Segmentation (test),HD95 mm LV/MY/RV,3.92 / 4.17 / 4.31
2,LVEF (test),MAE vs expert / vs ref (EF%),4.38 / 7.17
3,RQ1 (test),"Kruskal-Wallis H, p, eps^2","21.99, 1.68e-05, 0.222"
4,RQ2 (test),"Spearman rho(artifact granule, Dice), p","-0.543, 5.37e-09"


saved -> results_summary.csv
